# EDA part E — revenue per session over time

Makes 13 figures in `../figures/`: one all-markets chart, plus US-vs-market levels and
indexed for each of the 6 non-pilot markets.

Plotting code and design rationale are in [`../../src/eda_viz.py`](../../src/eda_viz.py).

In [ ]:
import sys
from pathlib import Path

# Resolve the repo root by walking up to the folder containing data/,
# so this notebook runs from any depth.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data" / "market_week_data.csv").exists())
sys.path.append(str(ROOT / "src"))

from eda_viz import load_panel, launch_week, plot_rps_all_markets, plot_rps_pairwise

df = load_panel()
LAUNCH = launch_week(df)

print(f"{df.shape[0]} rows x {df.shape[1]} cols")
print(f"{df['market'].nunique()} markets, {df['week_index'].nunique()} weeks")
print(f"advisor live in the US from {LAUNCH:%Y-%m-%d}")

## 1. All markets, US emphasised

The shaded band uses `digital_feature_available`, not `post_launch` — `post_launch` is
1 for all seven markets and would imply every market was treated.

In [ ]:
fig, ax = plot_rps_all_markets(df, savepath=str(ROOT / "figures/eda/rps_all_markets.png"))

## 2. Pairwise, US vs each non-pilot market

**Levels** = raw values, so the gap between markets stays visible. **Indexed** = each
market rebased to its own pre-launch mean = 100, which removes the level gap so you can
compare trends. Show levels first, then indexed.

> ⚠️ Indexed is a claim about *percent* change; a DiD on raw `revenue_per_session`
> assumes *absolute* parallel trends. If indexed is the parallel-trends argument, the
> model probably wants `log(revenue_per_session)`.

In [ ]:
from pathlib import Path

FIGDIR = ROOT / "figures/eda"
FIGDIR.mkdir(parents=True, exist_ok=True)

non_pilot = sorted(m for m in df["market"].unique() if m != "United States")
print(f"{len(non_pilot)} non-pilot markets: {', '.join(non_pilot)}\n")

written = []
for market in non_pilot:
    slug = market.lower().replace(" ", "_")
    for view, indexed in [("levels", False), ("indexed", True)]:
        path = FIGDIR / f"rps_us_vs_{slug}_{view}.png"
        plot_rps_pairwise(df, market, index_to_pre=indexed, savepath=str(path))
        written.append(path.name)

print(f"wrote {len(written)} figures:")
for name in written:
    print(" ", name)